# Arctic Wolf Data Retrieval Service API - Interactive Investigation

This notebook provides a comprehensive guide to exploring and using the Arctic Wolf Data Retrieval Service API.

## Overview
- **API Version**: 1.0.0-beta
- **Base URL**: https://data-retrieval-service-prod.managedgw.us001-prod.arcticwolf.net
- **Authentication**: PAK (Personal Access Key) Bearer Token
- **Python Version**: 3.9.6

## What You'll Learn
1. Setup and authentication
2. Discover available data sources
3. Explore predefined queries
4. Execute queries with different operators
5. Handle pagination and errors
6. Analyze results

## Resources
- [Customer Documentation](https://docs.arcticwolf.com/en/developer-and-oem/data-retrieval-api/arctic-wolf-data-retrieval-api)
- [Internal FAQ](https://arcticwolf.atlassian.net/wiki/spaces/PPM/pages/6273826895/Internal+Use+FAQ+Pertaining+to+Data+Retrieval+API)

## 1. Setup and Authentication

First, let's set up our environment and configure authentication.

In [1]:
import requests
import json
from datetime import datetime, timedelta, timezone
import pandas as pd
import time
import dotenv
import os
from pathlib import Path

# Load environment variables from .env
dotenv_path = Path.cwd() / ".env"
print(f"Loading environment from: {dotenv_path}")
dotenv.load_dotenv(dotenv_path, override=True)

# Configuration
BASE_URL = os.getenv("BASE_URL")
ORGANIZATION_ID = os.getenv("ORGANIZATION_ID")
PAK_TOKEN = os.getenv("PAK_TOKEN")

if PAK_TOKEN:
    PAK_TOKEN = PAK_TOKEN.strip()

print(f"Loaded BASE_URL repr: {repr(BASE_URL)}")
print(f"Loaded ORGANIZATION_ID repr: {repr(ORGANIZATION_ID)}")
print(f"Loaded PAK_TOKEN sections: {PAK_TOKEN.count('.') if PAK_TOKEN else 'None'}")

if not BASE_URL or not ORGANIZATION_ID or not PAK_TOKEN:
    raise ValueError(
        "Missing required environment variables. Please set BASE_URL, ORGANIZATION_ID, and PAK_TOKEN in a .env file."
    )

# Headers for authentication
headers = {
    "Authorization": f"Bearer {PAK_TOKEN}",
    "Content-Type": "application/json"
}

print("✅ Configuration loaded")
print(f"Base URL: {BASE_URL}")
print(f"Organization ID: {ORGANIZATION_ID}")
print(f"PAK Token: {'*' * (len(PAK_TOKEN)-4) + PAK_TOKEN[-4:] if len(PAK_TOKEN) > 4 else '****'}")



Loading environment from: /workspaces/Data_Service_API/.env
Loaded BASE_URL repr: 'https://data-retrieval-service-prod.managedgw.us001-prod.arcticwolf.net'
Loaded ORGANIZATION_ID repr: 'samplecollp'
Loaded PAK_TOKEN sections: 2
✅ Configuration loaded
Base URL: https://data-retrieval-service-prod.managedgw.us001-prod.arcticwolf.net
Organization ID: samplecollp
PAK Token: ***************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************

In [2]:
# Reusable helper functions

def safe_api_call(func, *args, **kwargs):
    """Wrapper for safe API calls with proper error handling."""
    try:
        return func(*args, **kwargs)
    except requests.exceptions.RequestException as e:
        print(f"🌐 Network error: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"📄 JSON parsing error: {e}")
        return None
    except Exception as e:
        print(f"⚠️ Unexpected error: {e}")
        return None


def execute_query(data_source, query_id, parameters, response_columns=None):
    """Execute a predefined query against the Data Retrieval API."""
    url = f"{BASE_URL}/api/v1beta/organizations/{ORGANIZATION_ID}/data-sources/{data_source}/predefined-queries/{query_id}/execute"
    payload = {"parameters": parameters}
    if response_columns:
        payload["response_columns"] = response_columns
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json()
    print(f"Error: {response.status_code} - {response.text}")
    return None


def execute_query_with_pagination(data_source, query_id, parameters, limit=100, max_results=1000):
    """Execute a query with pagination support."""
    all_results = []
    offset = 0
    columns = None
    print(f"🔄 Starting paginated query (limit={limit}, max_results={max_results})")
    while len(all_results) < max_results:
        paginated_params = parameters.copy()
        paginated_params.extend([
            {"name": "limit", "value": limit},
            {"name": "offset", "value": offset}
        ])
        result = execute_query(data_source, query_id, paginated_params)
        if not result or not result['results']:
            print(f"📄 No more results at offset {offset}")
            break
        all_results.extend(result['results'])
        columns = result['columns']
        if len(result['results']) < limit:
            print(f"📄 Reached end of results (got {len(result['results'])} < {limit})")
            break
        offset += limit
        print(f"📊 Retrieved {len(all_results)} results so far...")
    return {'columns': columns if columns else [], 'results': all_results}


def timed_query_execution(data_source, query_id, parameters):
    """Execute a query and print timing metrics."""
    print(f"⏱️ Starting timed query execution...")
    start = time.time()
    result = execute_query(data_source, query_id, parameters)
    duration = time.time() - start
    if result:
        print(f"\n⏱️ Performance Metrics:")
        print(f"  • Execution time: {duration:.2f} seconds")
        print(f"  • Results returned: {len(result['results'])} rows")
        if result['results']:
            print(f"  • Average time per row: {duration / len(result['results']) * 1000:.2f} ms")
            if result['columns']:
                estimated_size = len(result['results']) * len(result['columns']) * 50
                print(f"  • Estimated data size: {estimated_size/1024:.2f} KB")
    return result


def build_query_parameters(start_time, end_time, **kwargs):
    """Build query parameters with time range and optional filter criteria."""
    params = [
        {"name": "start_time", "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")},
        {"name": "end_time", "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")}
    ]
    for name, (operator, value) in kwargs.items():
        params.append({"name": name, "comparisonOperator": operator, "value": value})
    return params


def analyze_query_results(result):
    """Display a quick summary for query results."""
    if not result or not result['results']:
        print("❌ No results to analyze")
        return None
    df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
    print("📊 Query Results Analysis")
    print("=" * 30)
    print(f"  • Total rows: {len(df):,}")
    print(f"  • Total columns: {len(df.columns)}")
    print(f"  • Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    if 'at_timestamp' in df.columns:
        try:
            df['at_timestamp'] = pd.to_datetime(df['at_timestamp'])
            print(f"  • Time range: {df['at_timestamp'].min()} to {df['at_timestamp'].max()}")
        except Exception:
            pass
    print(df.head(3).to_string(index=False))
    return df


## 2. Discover Data Sources

Let's explore what data sources are available and examine their schemas.

In [3]:
# List all data sources
def list_data_sources():
    url = f"{BASE_URL}/api/v1beta/organizations/{ORGANIZATION_ID}/data-sources"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Get data source schema
def get_data_source_schema(data_source):
    url = f"{BASE_URL}/api/v1beta/organizations/{ORGANIZATION_ID}/data-sources/{data_source}/schema"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching schema:")
        print(f"  URL: {url}")
        print(f"  Status: {response.status_code}")
        print(f"  Response headers: {response.headers}")
        print(f"  Response text: {response.text}")
        return None

# Execute discovery
print("🔍 Discovering data sources...")
data_sources = list_data_sources()

if data_sources:
    print(f"\n📊 Available Data Sources ({len(data_sources)}):")
    for ds in data_sources:
        print(f"  • {ds['name']} (timeout: {ds['timeout']})")
else:
    print("❌ No data sources found or error occurred")

🔍 Discovering data sources...

📊 Available Data Sources (1):
  • observations (timeout: 60s)


In [4]:
# Get schema for observations data source
if data_sources:
    print("🔍 Getting schema for 'observations' data source...")
    schema = get_data_source_schema("observations")
    
    if schema:
        print(f"\n📋 Observations Schema ({len(schema['fields'])} fields):")
        
        # Create a DataFrame for better display
        schema_df = pd.DataFrame([
            {
                'Field': field['name'],
                'Type': field['type'],
                'Nullable': field['nullable'],
                'Description': field.get('description', 'No description')[:50] + '...' if field.get('description', '') else 'No description'
            }
            for field in schema['fields'][:15]  # Show first 15 fields
        ])
        
        print(schema_df.to_string(index=False))
        
        if len(schema['fields']) > 15:
            print(f"\n... and {len(schema['fields']) - 15} more fields")
    else:
        print("❌ Could not retrieve schema")

🔍 Getting schema for 'observations' data source...

📋 Observations Schema (236 fields):
                      Field      Type  Nullable                                           Description
               at_timestamp date_time     False         Date and time when the event was recorded....
                      @type    string     False              Type identifier for the event record....
   ad.event.auth.logon_type   integer     False Windows logon type code associated with the Active...
  ad.event.auth.result.code    string     False Result code returned by the Active Directory authe...
              ad.event.code    string     False Windows Event ID code for the Active Directory eve...
   ad.event.origin.username    string     False Username of the account that initiated the Active ...
 ad.event.target.group.name    string     False Name of the Active Directory group that was the ta...
   ad.event.target.username    string     False Username of the account that was the target of t

## 3. Explore Predefined Queries

Now let's discover what predefined queries are available and examine their parameters.

In [5]:
# List predefined queries for a data source
def list_predefined_queries(data_source):
    url = f"{BASE_URL}/api/v1beta/organizations/{ORGANIZATION_ID}/data-sources/{data_source}/predefined-queries"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Describe a specific query
def describe_query(data_source, query_id):
    url = f"{BASE_URL}/api/v1beta/organizations/{ORGANIZATION_ID}/data-sources/{data_source}/predefined-queries/{query_id}"
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Explore queries
print("🔍 Discovering predefined queries...")
queries = list_predefined_queries("observations")

if queries:
    print(f"\n📝 Available Predefined Queries ({len(queries['queries'])}):")
    for i, query in enumerate(queries['queries'], 1):
        print(f"  {i}. {query['name']}")
        print(f"     {query['description']}")
        print()
else:
    print("❌ No queries found or error occurred")

🔍 Discovering predefined queries...

📝 Available Predefined Queries (6):
  1. observations-by-login-status
     Find all observations based on login status

  2. observations-by-ip-address
     Find all observations related to a specific IP address

  3. observations-by-user
     Find all observations related to a specific user

  4. observations-by-hostname
     Find all observations related to a specific hostname

  5. observations-by-domain
     Find all observations related to a specific domain

  6. observations-by-event-code
     Find all observations by event code



In [6]:
# Ensure query helper functions are defined before using them
if 'list_predefined_queries' not in globals():
    print("🔍 Defining query helper functions...")

    def list_predefined_queries(data_source):
        url = f"{BASE_URL}/api/v1beta/organizations/{ORGANIZATION_ID}/data-sources/{data_source}/predefined-queries"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error: {response.status_code} - {response.text}")
            return None

    def describe_query(data_source, query_id):
        url = f"{BASE_URL}/api/v1beta/organizations/{ORGANIZATION_ID}/data-sources/{data_source}/predefined-queries/{query_id}"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Error: {response.status_code} - {response.text}")
            return None

if 'queries' not in globals() or queries is None:
    print("🔍 Discovering predefined queries...")
    queries = list_predefined_queries("observations")


# Get detailed description of IP address query
if queries:
    print("🔍 Getting detailed description for 'observations-by-ip-address' query...")
    ip_query_desc = describe_query("observations", "observations-by-ip-address")
    
    if ip_query_desc:
        print(f"\n📋 Query: {ip_query_desc['name']}")
        print(f"Description: {ip_query_desc['description']}")
        
        print(f"\n🔧 Parameters ({len(ip_query_desc['parameters'])}):")
        for param in ip_query_desc['parameters']:
            required = "✅ Required" if param['mandatory'] else "⚪ Optional"
            array_support = " (Array supported)" if param.get('canBeArray') else ""
            default = f" [Default: {param['default']}]" if param.get('default') is not None else ""
            
            print(f"  • {param['name']} ({param['type']}) - {required}{array_support}{default}")
            print(f"    {param['description']}")
            
            if param.get('allowed_operators'):
                print(f"    Operators: {', '.join(param['allowed_operators'])}")
            
            if param.get('validators'):
                validators = [v['type'] for v in param['validators']]
                print(f"    Validators: {', '.join(validators)}")
            print()
        
        # Show default columns
        if ip_query_desc.get('default_columns'):
            print(f"📊 Default Return Columns ({len(ip_query_desc['default_columns'])}):")
            for col in ip_query_desc['default_columns']:
                nullable = " (nullable)" if col.get('nullable') else ""
                print(f"  • {col['name']} ({col['type']}){nullable}")
                if col.get('description'):
                    print(f"    {col['description']}")
    else:
        print("❌ Could not retrieve query description")

🔍 Getting detailed description for 'observations-by-ip-address' query...

📋 Query: observations-by-ip-address
Description: Find all observations related to a specific IP address

🔧 Parameters (5):
  • start_time (timestamp) - ✅ Required
    The start of the query period (inclusive)
    Validators: iso_utc_timestamp

  • end_time (timestamp) - ✅ Required
    The end of the query period (exclusive)
    Validators: iso_utc_timestamp

  • limit (integer) - ⚪ Optional [Default: 100]
    How many results to return in a page
    Validators: positive, max

  • offset (integer) - ⚪ Optional [Default: 0]
    Offset describing where to return the results from across pages
    Validators: non_negative

  • ip_address (string) - ✅ Required (Array supported)
    The IP address to search for in the observations
    Operators: eq, in, contains
    Validators: not_blank, max_length

📊 Default Return Columns (12):
  • at_timestamp (date_time)
    Date and time when the event was recorded.
  • client.byt

## 4. Execute Queries

Now let's execute some queries with different parameters and operators.

In [7]:
# Execute a predefined query
def execute_query(data_source, query_id, parameters, response_columns=None):
    url = f"{BASE_URL}/api/v1beta/organizations/{ORGANIZATION_ID}/data-sources/{data_source}/predefined-queries/{query_id}/execute"
    
    payload = {
        "parameters": parameters
    }
    
    if response_columns:
        payload["response_columns"] = response_columns
    
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Set up time range for queries (last 24 hours) using timezone-aware UTC datetimes
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=24)

print(f"🕐 Query time range:")
print(f"  Start: {start_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"  End:   {end_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")

🕐 Query time range:
  Start: 2026-07-07 17:03:53 UTC
  End:   2026-07-08 17:03:53 UTC


In [8]:
# Example 1: Search by single IP address
print("🔍 Example 1: Single IP Address Search")
print("=" * 50)

parameters = [
    {
        "name": "start_time",
        "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "end_time",
        "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "ip_address",
        "comparisonOperator": "EQ", 
        "value": "10.171.170.112"
    }
]

print(f"Searching for IP: 10.171.170.112")
result = execute_query("observations", "observations-by-ip-address", parameters)

if result:
    print(f"\n✅ Found {len(result['results'])} observations")
    
    if result['results']:
        # Convert to DataFrame for better display
        df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
        print(f"\n📊 Sample Results (showing first 5 rows):")
        print(df.head().to_string(index=False))
        
        print(f"\n📈 Column Info:")
        for col in result['columns']:
            print(f"  • {col['name']} ({col['type']})")
    else:
        print("No results found for this IP address")
else:
    print("❌ Query execution failed")

🔍 Example 1: Single IP Address Search
Searching for IP: 10.171.170.112

✅ Found 100 observations

📊 Sample Results (showing first 5 rows):
            at_timestamp      client.ip  client.port client.geo.country_name client.geo.city_name  client.bytes host.os.family     server.ip  server.port  server.bytes server.geo.country_name server.geo.city_name
2026-07-07T17:03:57.469Z 10.171.170.112          NaN                     NaN                  NaN           NaN            NaN           NaN          NaN           NaN                     NaN                  NaN
2026-07-07T17:04:08.281Z 10.171.170.112      60059.0                     NaN                  NaN           NaN            NaN 10.171.170.11         53.0           NaN                     NaN                  NaN
2026-07-07T17:04:08.281Z 10.171.170.112      60059.0                     NaN                  NaN           NaN        Windows 10.171.170.11         53.0           NaN                     NaN                  NaN
2026-07-0

In [9]:
# Example 2: Search by multiple IP addresses using IN operator
print("🔍 Example 2: Multiple IP Addresses (IN operator)")
print("=" * 50)

# Build the parameters explicitly for this example so the cell can run independently
multi_ip_parameters = [
    {
        "name": "start_time",
        "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "end_time",
        "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "ip_address",
        "comparisonOperator": "IN",
        "value": ["10.171.170.112", "10.171.170.105", "10.171.170.107"]
    }
]

ip_list = multi_ip_parameters[2]["value"]
print(f"Searching for IPs: {', '.join(ip_list)}")

result = execute_query("observations", "observations-by-ip-address", multi_ip_parameters)

if result:
    print(f"\n✅ Found {len(result['results'])} observations for multiple IPs")
    
    if result['results']:
        df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
        
        # Analyze results by IP if client.ip column exists
        if 'client.ip' in df.columns:
            ip_counts = df['client.ip'].value_counts()
            print(f"\n📊 Results by IP address:")
            for ip, count in ip_counts.items():
                print(f"  • {ip}: {count} observations")
        
        print(f"\n📋 Sample Results:")
        print(df.head(3).to_string(index=False))
    else:
        print("No results found for these IP addresses")
else:
    print("❌ Query execution failed")

🔍 Example 2: Multiple IP Addresses (IN operator)
Searching for IPs: 10.171.170.112, 10.171.170.105, 10.171.170.107

✅ Found 100 observations for multiple IPs

📊 Results by IP address:
  • 10.171.170.105: 53 observations
  • 10.171.170.112: 31 observations
  • 10.171.170.107: 16 observations

📋 Sample Results:
            at_timestamp      client.ip  client.port client.geo.country_name client.geo.city_name  client.bytes host.os.family     server.ip  server.port  server.bytes server.geo.country_name server.geo.city_name
2026-07-07T17:03:56.958Z 10.171.170.105      48178.0                     NaN                  NaN           NaN        Unknown 10.171.170.11         53.0           NaN                     NaN                  NaN
2026-07-07T17:03:56.958Z 10.171.170.105      48178.0                     NaN                  NaN           NaN        Windows 10.171.170.11         53.0           NaN                     NaN                  NaN
2026-07-07T17:03:56.963Z 10.171.170.105      41684

In [16]:
# Example 3: Search with CONTAINS operator (hostname)
print("🔍 Example 3: Hostname Contains Search")
print("=" * 50)

hostname_params = [
    {
        "name": "start_time",
        "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "end_time",
        "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "host_name",
        "comparisonOperator": "CONTAINS",
        "value": "DESKTOP"
    }
]

print(f"Searching for hostnames containing: 'DESKTOP'")
result = execute_query("observations", "observations-by-hostname", hostname_params)

if result:
    print(f"\n✅ Found {len(result['results'])} observations with hostname containing 'DESKTOP'")
    
    if result['results']:
        df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
        print(f"\n📋 Sample Results:")
        print(df.head(3).to_string(index=False))
    else:
        print("No results found for hostnames containing 'DESKTOP'")
else:
    print("❌ Query execution failed")

🔍 Example 3: Hostname Contains Search
Searching for hostnames containing: 'DESKTOP'

✅ Found 100 observations with hostname containing 'DESKTOP'

📋 Sample Results:
            at_timestamp      client.ip client.port server.ip server.port host.hostname event.code ad.event.origin.username                        ad.event.title event.outcome
2026-07-07T17:05:44.451Z 10.171.170.115        None      None        None      DESKTOP7       7040                      NaN       awn-agent-win-security-unmapped           NaN
2026-07-07T17:11:21.375Z 10.171.170.102        None      None        None      DESKTOP2       4624                DESKTOP2$ An account was successfully logged on       success
2026-07-07T17:11:21.721Z 10.171.170.102        None      None        None      DESKTOP2       4624                DESKTOP2$ An account was successfully logged on       success


## 5. Advanced Usage & Best Practices

Let's explore advanced features like pagination, error handling, and performance monitoring.

In [11]:
# Advanced query execution with error handling and pagination
def execute_query_with_pagination(data_source, query_id, parameters, limit=100, max_results=1000):
    """
    Execute query with automatic pagination to retrieve more results
    """
    all_results = []
    offset = 0
    columns = None
    
    print(f"🔄 Starting paginated query (limit={limit}, max_results={max_results})")
    
    while len(all_results) < max_results:
        # Add pagination parameters
        paginated_params = parameters.copy()
        paginated_params.extend([
            {
                "name": "limit",
                "value": limit
            },
            {
                "name": "offset", 
                "value": offset
            }
        ])
        
        try:
            result = execute_query(data_source, query_id, paginated_params)
            
            if not result or not result['results']:
                print(f"📄 No more results at offset {offset}")
                break
                
            all_results.extend(result['results'])
            columns = result['columns']
            
            # If we got fewer results than the limit, we've reached the end
            if len(result['results']) < limit:
                print(f"📄 Reached end of results (got {len(result['results'])} < {limit})")
                break
                
            offset += limit
            print(f"📊 Retrieved {len(all_results)} results so far...")
            
        except Exception as e:
            print(f"❌ Error during pagination: {e}")
            break
    
    return {
        'columns': columns if columns else [],
        'results': all_results
    }

# Error handling wrapper
def safe_api_call(func, *args, **kwargs):
    """
    Wrapper for safe API calls with proper error handling
    """
    try:
        return func(*args, **kwargs)
    except requests.exceptions.RequestException as e:
        print(f"🌐 Network error: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"📄 JSON parsing error: {e}")
        return None
    except Exception as e:
        print(f"⚠️ Unexpected error: {e}")
        return None

print("✅ Advanced functions loaded")

✅ Advanced functions loaded


In [12]:
# Performance monitoring
def timed_query_execution(data_source, query_id, parameters):
    """
    Execute query with timing information
    """
    print(f"⏱️ Starting timed query execution...")
    start_time = time.time()
    
    result = execute_query(data_source, query_id, parameters)
    
    end_time = time.time()
    
    if result:
        execution_time = end_time - start_time
        print(f"\n⏱️ Performance Metrics:")
        print(f"  • Execution time: {execution_time:.2f} seconds")
        print(f"  • Results returned: {len(result['results'])} rows")
        
        if result['results']:
            avg_time_per_row = execution_time / len(result['results'])
            print(f"  • Average time per row: {avg_time_per_row*1000:.2f} ms")
            
            # Estimate data size
            if result['columns']:
                estimated_size = len(result['results']) * len(result['columns']) * 50  # rough estimate
                print(f"  • Estimated data size: {estimated_size/1024:.2f} KB")
    
    return result

# Test performance with a simple query
print("🚀 Testing query performance...")
perf_result = timed_query_execution("observations", "observations-by-ip-address", parameters)

🚀 Testing query performance...
⏱️ Starting timed query execution...

⏱️ Performance Metrics:
  • Execution time: 1.13 seconds
  • Results returned: 100 rows
  • Average time per row: 11.33 ms
  • Estimated data size: 58.59 KB


In [13]:
# Data analysis helpers
def analyze_query_results(result):
    """
    Analyze query results and provide insights
    """
    if not result or not result['results']:
        print("❌ No results to analyze")
        return None
    
    df = pd.DataFrame(result['results'], columns=[col['name'] for col in result['columns']])
    
    print("📊 Query Results Analysis")
    print("=" * 30)
    print(f"📈 Dataset Overview:")
    print(f"  • Total rows: {len(df):,}")
    print(f"  • Total columns: {len(df.columns)}")
    print(f"  • Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
    
    # Show data types
    print(f"\n🏷️ Column Types:")
    for col in result['columns']:
        sample_values = df[col['name']].dropna().head(3).tolist()
        sample_str = f" (e.g., {sample_values})" if sample_values else ""
        print(f"  • {col['name']}: {col['type']}{sample_str}")
    
    # Show sample data
    print(f"\n📋 Sample Data (first 3 rows):")
    print(df.head(3).to_string(index=False))
    
    # Basic statistics for numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        print(f"\n📊 Numeric Column Statistics:")
        print(df[numeric_cols].describe().round(2))
    
    return df

# Example usage of advanced pagination
print("🔍 Advanced Query Execution Example")
print("=" * 40)

# Use the advanced pagination function
large_result = execute_query_with_pagination(
    "observations", 
    "observations-by-ip-address", 
    parameters, 
    limit=50, 
    max_results=200
)

if large_result and large_result['results']:
    df = analyze_query_results(large_result)
    
    # Additional time-based analysis if timestamp column exists
    if df is not None and 'at_timestamp' in df.columns:
        try:
            df['at_timestamp'] = pd.to_datetime(df['at_timestamp'])
            print(f"\n🕐 Time Analysis:")
            print(f"  • Time range: {df['at_timestamp'].min()} to {df['at_timestamp'].max()}")
            print(f"  • Duration: {df['at_timestamp'].max() - df['at_timestamp'].min()}")
            
            # Group by hour
            hourly_counts = df.groupby(df['at_timestamp'].dt.hour).size()
            if len(hourly_counts) > 0:
                print(f"\n📅 Observations by hour:")
                for hour, count in hourly_counts.head(10).items():
                    print(f"  • Hour {hour:02d}: {count} observations")
        except Exception as e:
            print(f"⚠️ Could not analyze timestamps: {e}")
else:
    print("❌ No results from paginated query")

🔍 Advanced Query Execution Example
🔄 Starting paginated query (limit=50, max_results=200)
📊 Retrieved 50 results so far...
📊 Retrieved 100 results so far...
Error: 502 - <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
</body>
</html>

📄 No more results at offset 100
📊 Query Results Analysis
📈 Dataset Overview:
  • Total rows: 100
  • Total columns: 12
  • Memory usage: 40.04 KB

🏷️ Column Types:
  • at_timestamp: date_time (e.g., ['2026-07-07T17:03:57.469Z', '2026-07-07T17:04:08.281Z', '2026-07-07T17:04:08.281Z'])
  • client.ip: string (e.g., ['10.171.170.112', '10.171.170.112', '10.171.170.112'])
  • client.port: integer (e.g., [60059.0, 60059.0, 55381.0])
  • client.geo.country_name: string (e.g., ['United States', 'Ireland', 'United States'])
  • client.geo.city_name: string (e.g., ['Redmond', 'Dublin', 'Boardman'])
  • client.bytes: integer (e.g., [1590.0, 1407.0, 606.0])
  • host.os.family: string (e.g., ['Windows', 'Windows', 'W

## 6. Experiment Zone

Use this section to experiment with different queries and parameters.

In [14]:
# Experiment with different queries
# Try different query types available in your organization

print("🧪 Experiment Zone - Try Your Own Queries!")
print("=" * 50)

# Example: Query by user
user_params = [
    {
        "name": "start_time",
        "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "end_time",
        "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    },
    {
        "name": "user",
        "comparisonOperator": "CONTAINS",
        "value": "admin"  # Change this to a user you want to search for
    }
]

print("Trying user-based query...")
user_result = safe_api_call(execute_query, "observations", "observations-by-user", user_params)

if user_result:
    print(f"✅ User query returned {len(user_result['results'])} results")
else:
    print("❌ User query failed or returned no results")

# Add your own experiments below:
# TODO: Try different query types
# TODO: Experiment with different operators
# TODO: Test different time ranges
# TODO: Try custom response_columns

🧪 Experiment Zone - Try Your Own Queries!
Trying user-based query...
✅ User query returned 100 results


In [15]:
# Custom query builder helper
def build_query_parameters(start_time, end_time, **kwargs):
    """
    Helper function to build query parameters easily
    """
    params = [
        {
            "name": "start_time",
            "value": start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        },
        {
            "name": "end_time",
            "value": end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
        }
    ]
    
    for param_name, (operator, value) in kwargs.items():
        params.append({
            "name": param_name,
            "comparisonOperator": operator,
            "value": value
        })
    
    return params

# Example usage of query builder
print("🛠️ Using Query Builder Helper")
print("=" * 30)

if "start_time" not in globals() or "end_time" not in globals():
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=24)

# Build a query for multiple domains
domain_params = build_query_parameters(
    start_time, 
    end_time,
    domain=("IN", ["example.com", "test.com", "demo.org"])
)

print("Built parameters for domain query:")
for param in domain_params:
    print(f"  • {param['name']}: {param.get('comparisonOperator', '')} {param['value']}")

# Try the domain query
domain_result = safe_api_call(execute_query, "observations", "observations-by-domain", domain_params)

if domain_result:
    print(f"\n✅ Domain query returned {len(domain_result['results'])} results")
else:
    print("\n❌ Domain query failed or returned no results")

🛠️ Using Query Builder Helper
Built parameters for domain query:
  • start_time:  2026-07-07T17:03:53Z
  • end_time:  2026-07-08T17:03:53Z
  • domain: IN ['example.com', 'test.com', 'demo.org']

✅ Domain query returned 0 results


## Summary

This notebook has covered:

1. **Authentication** - Setting up PAK token authentication
2. **Discovery** - Finding available data sources and their schemas
3. **Query Exploration** - Understanding predefined queries and their parameters
4. **Query Execution** - Running queries with different operators (EQ, IN, CONTAINS)
5. **Advanced Features** - Pagination, error handling, and performance monitoring
6. **Data Analysis** - Processing and analyzing query results

### Next Steps

- Explore other available queries in your organization
- Experiment with different time ranges and parameters
- Build custom analysis workflows using the query results
- Integrate with other data analysis tools and workflows

### Resources

- [Arctic Wolf Data Retrieval API Documentation](https://docs.arcticwolf.com/en/developer-and-oem/data-retrieval-api/arctic-wolf-data-retrieval-api)
- [Internal FAQ](https://arcticwolf.atlassian.net/wiki/spaces/PPM/pages/6273826895/Internal+Use+FAQ+Pertaining+to+Data+Retrieval+API)

Happy querying! 🚀